# Factory Functions in JavaScript

A **factory function** is a regular function that returns a new object without using the `new` keyword. It acts as a template or manufacturing blueprint, letting you generate multiple object instances with similar structure but distinct data.

> See also: [[javascript-oop]] · [[javascript-prototypes]]

---

## Basic Example

Instead of duplicating object literals by hand, a factory centralizes the creation logic:

```javascript
function createUser(name, role) {
  return {
    name: name,
    role: role,
    greet() {
      console.log(`Hello, my name is ${this.name} and I am an ${this.role}.`);
    }
  };
}

const user1 = createUser('Alice', 'Admin');
const user2 = createUser('Bob', 'Editor');

user1.greet(); // "Hello, my name is Alice and I am an Admin."
```

**Naming convention:** prefix with `create…` or `make…`. Constructor functions and classes use `PascalCase`; factories use `camelCase` so the call site tells you no `new` is needed.

---

## Core Advantages

### No `new` or `this` complexities

You avoid the pitfalls of constructor functions and classes, which misbehave when a developer forgets `new`:

```javascript
function User(name) { this.name = name; }

const u = User("Alice");   // no `new` — returns undefined,
                           // and in sloppy mode leaks `name` onto globalThis
```

Classes at least throw a `TypeError` here, but factories make the problem structurally impossible.

### True data privacy via closures

Local variables inside the factory are unreachable from outside — enforced by scope, not convention.

### High flexibility

Unlike a fixed class hierarchy, a factory can decide at runtime what shape of object to return.

---

## Implementing Private Variables with Closures

```javascript
function createBankAccount(owner, initialBalance) {
  // Private variable, completely hidden from the outside scope
  let balance = initialBalance;

  return {
    owner,
    getBalance() {
      return balance;
    },
    deposit(amount) {
      if (amount > 0) balance += amount;
    }
  };
}

const account = createBankAccount('Charlie', 1000);
console.log(account.balance);      // undefined (hidden!)
console.log(account.getBalance()); // 1000
account.deposit(500);
console.log(account.getBalance()); // 1500
```

### Private *functions* too

Anything declared in the factory scope but not returned is private:

```javascript
function createValidator(rules) {
  // private helper — never exposed
  const normalize = (v) => String(v).trim().toLowerCase();

  return {
    check(value) {
      const clean = normalize(value);
      return rules.every(rule => rule(clean));
    }
  };
}
```

---

## Modern Syntax Enhancements (ES6+)

Property shorthand and implicit arrow returns keep factories terse. Note the parentheses around the object literal — without them, `{` is parsed as a function body.

```javascript
const createRobot = (name, type) => ({
  name,
  type,
  identify: () => console.log(`Robot ${name} online.`)
});

const drone = createRobot('Flyer-1', 'Drone');
```

### ⚠️ Arrow methods close over parameters, not `this`

The example above works, but only because `identify` reads the closed-over `name` **parameter**, not `this.name`. Swap in `this.name` and it breaks:

```javascript
const createRobot = (name) => ({
  name,
  bad:  () => console.log(this.name),        // ❌ `this` is not the object
  good() { console.log(this.name); }         // ✅ method shorthand
});
```

There's a subtle behavioural difference too: the closure version captures the *original argument* forever, while the `this` version reads the current property.

```javascript
drone.name = 'Flyer-2';
drone.identify();  // still logs "Robot Flyer-1 online." — closure captured the parameter
```

Pick deliberately: closure capture for immutable private state, `this` for live property reads.

---

## Options Object + Defaults

Once a factory takes more than two or three parameters, switch to a destructured options object. It kills argument-order bugs and self-documents at the call site.

```javascript
function createServer({
  host = 'localhost',
  port = 3000,
  logger = console,
  middleware = []
} = {}) {
  return {
    start() {
      logger.log(`Listening on ${host}:${port}`);
    },
    use(fn) {
      middleware.push(fn);
      return this;   // enable chaining
    }
  };
}

createServer({ port: 8080 }).start();
```

### Validation and guard clauses

A factory is a natural place to fail fast, before an object even exists:

```javascript
function createUser({ name, email }) {
  if (!name?.trim()) throw new TypeError('name is required');
  if (!email?.includes('@')) throw new TypeError('invalid email');

  return Object.freeze({ name, email });   // shallow-immutable result
}
```

---

## Factories That Return Different Shapes

This is where factories genuinely beat classes — the return type is a runtime decision.

```javascript
function createStorage(env) {
  if (env === 'production') {
    return { save: (k, v) => redisClient.set(k, v) };
  }
  if (env === 'test') {
    const store = new Map();
    return { save: (k, v) => store.set(k, v) };
  }
  return { save: (k, v) => console.log('DEV save', k, v) };
}
```

The caller gets one interface and never knows which implementation it received. Doing this with classes requires an extra factory layer anyway.

---

## Composition with Factories

Instead of an inheritance tree, build objects by merging capability slices:

```javascript
const canFly    = (state) => ({ fly:  () => console.log(`${state.name} flies`) });
const canSwim   = (state) => ({ swim: () => console.log(`${state.name} swims`) });
const canQuack  = (state) => ({ quack: () => console.log(`${state.name} quacks`) });

function createDuck(name) {
  const state = { name };
  return Object.assign({}, state, canFly(state), canSwim(state), canQuack(state));
}

const donald = createDuck('Donald');
donald.fly();
donald.quack();
```

Adding or removing an ability is a one-line change, with no base class to renegotiate.

---

## The Big Practical Win: Destructuring Survives

Factory methods keep working when pulled off the object, because they close over state instead of depending on `this`:

```javascript
const account = createBankAccount('Charlie', 1000);
const { deposit, getBalance } = account;
deposit(500);
getBalance();   // 1500 ✅ works

class Account {
  #balance = 1000;
  deposit(n) { this.#balance += n; }
}
const { deposit: d } = new Account();
d(500);         // ❌ TypeError — `this` is undefined
```

This matters constantly in React hooks, event handlers, and anywhere you pass a method as a callback.

---

## Downsides and Trade-offs

The original framing is one-sided; these are the real costs.

### 1. Memory: methods are duplicated per instance

Every call creates a fresh set of function objects. Classes create them once on the prototype.

```javascript
const a = createUser('A', 'Admin');
const b = createUser('B', 'Editor');
a.greet === b.greet;   // false — two separate function objects
```

For a few hundred objects this is irrelevant. For tens of thousands (rendering rows, game entities, parsing large datasets) it's measurable. Mitigate by sharing a methods object as the prototype:

```javascript
const userMethods = {
  greet() { console.log(`Hi, ${this.name}`); }
};

function createUser(name, role) {
  const obj = Object.create(userMethods);  // methods shared, not copied
  obj.name = name;
  obj.role = role;
  return obj;
}
```

You trade closure privacy for prototype sharing — you can't have both for the same member.

### 2. `instanceof` doesn't work

```javascript
const u = createUser('Alice', 'Admin');
u instanceof createUser;   // false, always
```

Use duck typing, a `type` tag field, or `Symbol.hasInstance` if you need runtime identity checks.

### 3. Harder to debug and inspect

Devtools show a generic `Object`, not `User`. Stack traces and logs are less informative. A `type` or `Symbol.toStringTag` property helps:

```javascript
return {
  [Symbol.toStringTag]: 'User',
  name, role
};
```

### 4. No native extension mechanism

There's no `extends`. You compose or wrap manually, which is more explicit but also more code for deep hierarchies.

### 5. Private state is invisible to serialization

`JSON.stringify(account)` gives `{"owner":"Charlie"}` — the balance is gone, because it never was a property. Add an explicit `toJSON()` if you need serialization.

---

## Factory Functions vs. Constructor Functions vs. ES6 Classes

| Feature | Factory Functions | Constructor Functions | ES6 Classes |
|---|---|---|---|
| Instantiation | `createObj()` | `new Obj()` | `new MyClass()` |
| Forgetting `new` | N/A — impossible | Silent bug | `TypeError` (safe) |
| Privacy | Native and complete via closures | `_` convention only | `#private` fields |
| Method storage | Per instance (unless you use `Object.create`) | Shared via prototype | Shared via prototype |
| `this` binding | Avoidable entirely | Fragile | Fragile |
| Survives destructuring | ✅ Yes | ❌ No | ❌ No |
| `instanceof` | ❌ No | ✅ Yes | ✅ Yes |
| Inheritance | Composition / `Object.create` | Manual prototype wiring | `extends` + `super` |
| Return shape | Dynamic at runtime | Fixed | Fixed |
| Devtools naming | Generic `Object` | Named | Named |

---

## Where Factories Shine in Node/Express

Factories are the natural fit for services and repositories with injected dependencies — testable without mocking libraries:

```javascript
// repository factory
export function createCampgroundRepo(db) {
  return {
    findById: (id) => db.collection('campgrounds').findOne({ _id: id }),
    create:   (data) => db.collection('campgrounds').insertOne(data)
  };
}

// in tests, inject a fake — no jest.mock needed
const repo = createCampgroundRepo(fakeDb);
```

Same idea as swapping a data source behind a Tableau workbook: the consumer codes against the interface, not the implementation.

**Rule of thumb:**
- **Factory** — you need closures/privacy, dynamic shapes, dependency injection, or callback-safe methods.
- **Class** — you need many instances, real inheritance, `instanceof`, or you're subclassing a built-in like `Error`.
- **Plain module object** — you need one collection of related functions, not instances at all.

---

## Quick Reference

```javascript
// Minimal
const make = (a, b) => ({ a, b });

// With privacy
function make(a) {
  let secret = a;
  return { get: () => secret };
}

// With shared methods (memory-efficient)
const proto = { greet() { return `Hi ${this.name}`; } };
const make = (name) => Object.assign(Object.create(proto), { name });

// Immutable result
const make = (a) => Object.freeze({ a });

// Composed
const make = (name) => Object.assign({}, canFly({ name }), canSwim({ name }));
```